# SHL Hiring Assessment 2026 — Grammar Scoring Engine (FINAL)
**Goal:** spoken audio (45–60 s) → grammar score in [0, 5]. Metrics: RMSE + Pearson.

## Report (summary)
*(v6: adds Whisper-medium encoder embeddings and 3-seed CV)*
**Approach.** Grammar depends on *what* is said (text) and partly *how* (fluency). Each clip is described by:
1. **Whisper-medium transcript → text features:** LanguageTool error rate, CoLA grammatical-acceptability (RoBERTa), GPT-2 log-perplexity, sentence-length stats, lexical diversity, fillers, repetitions. Plus a **sentence embedding** (all-mpnet-base-v2) of the transcript.
2. **Audio features:** Whisper-medium encoder embeddings (layers 12/18/24), wav2vec2-base mean/std pooled embeddings (layer 7, plus layers 3/5/7/9/11 mean), and speech-rate / pause / loudness statistics (librosa).

**Models.** Ridge / SVR / Huber / RandomForest / Gradient-Boosting base learners on different feature views → 5-fold out-of-fold (OOF) predictions averaged over two split seeds → non-negative linear stacker → clip to [0, 5]. The notebook **automatically compares the full blend with the original 4-model blend** by nested-CV RMSE and keeps the better one.

**Zero-score clips.** 37/769 training clips have label 0 (outside the 1–5 rubric, likely unusable audio). `DROP_ZEROS=True` (default) trains without them; `False` keeps them and learns a zero-gate classifier.

**Evaluation.** Training RMSE (required), plus honest 5-fold OOF RMSE / Pearson. **Limitation:** Whisper may silently fix grammar mistakes; wav2vec2 and fluency features partly compensate.

**Final version.** Best leaderboard result (0.3800): Whisper-medium transcripts + a second set of *prompted* transcripts (keeps fillers / errors), LanguageTool + CoLA + GPT-2 + fluency features, wav2vec2 (layer 7 + multi-layer), Whisper-medium encoder embeddings, sentence embeddings, 13 base models, non-negative stacking. Zero-score clips are excluded from training.

In [ ]:
!pip install language-tool-python -q

In [ ]:
import os, re, glob, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, librosa
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else -1
dev = 'cuda' if DEVICE == 0 else 'cpu'
SR = 16000
print('GPU available:', torch.cuda.is_available())

hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
ROOT = os.path.dirname(hits[0]) if hits else '.'
print('ROOT =', ROOT)

## 1. Load data

In [ ]:
train    = pd.read_csv(f'{ROOT}/train.csv')
test_csv = pd.read_csv(f'{ROOT}/test.csv')
sub      = pd.read_csv(f'{ROOT}/sample_submission.csv')
FNAME, LABEL = 'filename', 'label'
stem = lambda s: os.path.splitext(os.path.basename(str(s)))[0]

def index_wavs(folder):                      # train/ and test/ reuse file names -> index per folder
    return {stem(p): p for p in glob.glob(f'{ROOT}/**/{folder}/**/*.wav', recursive=True)}
train_map, test_map = index_wavs('train'), index_wavs('test')
if not train_map and not test_map:
    train_map = test_map = {stem(p): p for p in glob.glob(f'{ROOT}/**/*.wav', recursive=True)}
print('wavs found -> train:', len(train_map), '| test:', len(test_map))

train['path'] = train[FNAME].map(lambda f: train_map.get(stem(f)))
test = pd.DataFrame({FNAME: test_csv[FNAME].values})           # the 216 IDs of test.csv
test['path'] = test[FNAME].map(lambda f: test_map.get(stem(f)))
assert train.path.notna().all() and test.path.notna().all(), 'some wav files missing'
print(len(train), len(test))

## 2. EDA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.countplot(x=train[LABEL].round(1), ax=ax[0]); ax[0].set_title('Label distribution')
sns.histplot([librosa.get_duration(path=p) for p in train.path[:100]], ax=ax[1], bins=15); ax[1].set_title('Duration (s), 100 train clips')
plt.tight_layout(); plt.show(); print(train[LABEL].describe())

### (Optional) re-use caches from an earlier notebook
If a previous notebook's *Output* is attached as Input, this copies its `cache_*` files so the slow steps are skipped. Harmless if nothing is attached.

In [ ]:
import glob, shutil
for f in glob.glob('/kaggle/input/**/cache_*', recursive=True):
    if not os.path.exists(os.path.basename(f)): shutil.copy(f, '.')
print([f for f in os.listdir('.') if f.startswith('cache_')])

## 3. Preprocessing + Whisper-medium transcription (memory-safe batches)

In [ ]:
from transformers import pipeline
def load(p):
    y, _ = librosa.load(p, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)          # strip leading/trailing silence
    return y

CACHE = 'cache_text_medium_final.csv'
if os.path.exists(CACHE):
    tx = pd.read_csv(CACHE).fillna('')
else:
    asr = pipeline('automatic-speech-recognition', model='openai/whisper-medium', device=DEVICE,
                   chunk_length_s=30, torch_dtype=torch.float16 if DEVICE == 0 else torch.float32)
    paths, texts, BS = pd.concat([train.path, test.path]).tolist(), [], 4
    for i in tqdm(range(0, len(paths), BS), desc='Whisper'):
        res = asr([{'raw': load(p), 'sampling_rate': SR} for p in paths[i:i+BS]], batch_size=BS,
                  generate_kwargs={'language': 'english', 'task': 'transcribe'})
        texts += [r['text'].strip() for r in res]
    tx = pd.DataFrame({'text': texts}); tx.to_csv(CACHE, index=False)
    del asr; torch.cuda.empty_cache()
train['text'] = tx.text.values[:len(train)]; test['text'] = tx.text.values[len(train):]
train[[FNAME, 'text', LABEL]].head()

## 4. Text features (LanguageTool, CoLA, GPT-2, fluency statistics)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
TXT_CACHE = 'cache_textfeats_medium_final.csv'
if os.path.exists(TXT_CACHE):
    TF = pd.read_csv(TXT_CACHE, index_col=0)
else:
    import language_tool_python
    try: tool = language_tool_python.LanguageTool('en-US')
    except Exception as e: tool = None; print('LanguageTool unavailable -> grammar_error_rate = 0 :', e)
    cola = pipeline('text-classification', model='textattack/roberta-base-CoLA', device=DEVICE, top_k=None, truncation=True, max_length=128)
    tok = AutoTokenizer.from_pretrained('gpt2'); lm = AutoModelForCausalLM.from_pretrained('gpt2').eval().to(dev)
    FILLERS = {'uh','um','umm','uhh','er','erm','ah','hmm'}
    def sentences(t):
        s = [x.strip() for x in re.split(r'(?<=[.!?])\s+', t) if len(x.split()) >= 2]
        return s or [t if t else 'empty']
    @torch.no_grad()
    def ppl(t):
        ids = tok(t, return_tensors='pt', truncation=True, max_length=512).input_ids.to(dev)
        if ids.shape[1] < 2: return 0.0
        return float(torch.exp(lm(ids, labels=ids).loss).clamp(max=5000).log())
    def acc(res):
        d = {r['label']: r['score'] for r in res}; return d.get('LABEL_1', d.get('acceptable', 0.0))
    def text_feats(t):
        w = re.findall(r"[a-zA-Z']+", t.lower()); n = max(len(w), 1)
        ss = sentences(t); sl = [len(s.split()) for s in ss]; p = [acc(r) for r in cola(ss)]
        bg = list(zip(w, w[1:]))
        return dict(grammar_error_rate=(len(tool.check(t)) / n) if tool else 0.0,
            n_words=n, n_sent=len(ss), avg_sent_len=np.mean(sl), max_sent_len=max(sl), std_sent_len=np.std(sl),
            ttr=len(set(w))/n, filler_rate=sum(x in FILLERS for x in w)/n,
            repeat_bigram=1 - len(set(bg))/max(len(bg), 1), immediate_repeat=sum(a == b for a, b in bg)/n,
            avg_word_len=np.mean([len(x) for x in w]) if w else 0,
            cola_mean=np.mean(p), cola_min=np.min(p), cola_frac_ok=np.mean(np.array(p) > 0.5),
            log_ppl=ppl(t), has_punct=float(bool(re.search(r'[.!?]', t))))
    TF = pd.DataFrame([text_feats(t) for t in tqdm(pd.concat([train.text, test.text]), desc='Text feats')])
    TF.to_csv(TXT_CACHE); del cola, lm; torch.cuda.empty_cache()
X_txt_tr, X_txt_te = TF.iloc[:len(train)].values, TF.iloc[len(train):].values
print(TF.shape)

## 5. Audio features (wav2vec2 + prosody) and extra embeddings

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor, AutoModel
fe = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base-960h')
w2v = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h', output_hidden_states=True).eval().to(dev)
LAYERS = (3, 5, 7, 9, 11)

@torch.no_grad()
def audio_feats(p):
    y = load(p)
    x = fe(y, sampling_rate=SR, return_tensors='pt').input_values.to(dev)
    hs = w2v(x).hidden_states
    h = hs[7][0]; emb = torch.cat([h.mean(0), h.std(0)]).cpu().numpy()                 # layer-7 mean+std
    ml = np.concatenate([hs[l][0].mean(0).cpu().numpy() for l in LAYERS])              # multi-layer mean
    iv = librosa.effects.split(y, top_db=30)
    sp = sum(e - s for s, e in iv)/SR; tot = len(y)/SR
    gaps = [(iv[i+1][0] - iv[i][1])/SR for i in range(len(iv) - 1)]
    pro = [tot, sp/max(tot, 1e-6), len(iv)/max(tot, 1e-6), np.mean(gaps) if gaps else 0, np.max(gaps) if gaps else 0,
           float(np.mean(librosa.feature.rms(y=y)))]
    return emb, np.array(pro), ml

AUD_CACHE = 'cache_audio_v5.npz'
if os.path.exists(AUD_CACHE):
    z = np.load(AUD_CACHE); EMB, PRO, WL = z['emb'], z['pro'], z['wl']
else:
    r = [audio_feats(p) for p in tqdm(pd.concat([train.path, test.path]), desc='Audio feats')]
    EMB, PRO, WL = np.stack([a for a, _, _ in r]), np.stack([b for _, b, _ in r]), np.stack([c for _, _, c in r])
    np.savez(AUD_CACHE, emb=EMB, pro=PRO, wl=WL)
del w2v; torch.cuda.empty_cache()

# transcript sentence embeddings
EMB_CACHE = 'cache_text_emb_v5.npy'
if os.path.exists(EMB_CACHE): TE = np.load(EMB_CACHE)
else:
    etok = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
    emod = AutoModel.from_pretrained('sentence-transformers/all-mpnet-base-v2').eval().to(dev)
    texts, out = pd.concat([train.text, test.text]).fillna('').tolist(), []
    with torch.no_grad():
        for i in range(0, len(texts), 16):
            b = etok(texts[i:i+16], padding=True, truncation=True, max_length=384, return_tensors='pt').to(dev)
            h = emod(**b).last_hidden_state; m = b['attention_mask'].unsqueeze(-1).float()
            out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    TE = np.vstack(out); np.save(EMB_CACHE, TE); del emod; torch.cuda.empty_cache()

n0 = len(train)
X_emb_tr, X_emb_te = EMB[:n0], EMB[n0:]; X_pro_tr, X_pro_te = PRO[:n0], PRO[n0:]
W_tr, W_te = WL[:n0], WL[n0:]; T_tr, T_te = TE[:n0], TE[n0:]
print(EMB.shape, PRO.shape, WL.shape, TE.shape)

### 5b. Whisper-medium ENCODER embeddings (strong speech-quality / proficiency signal; layers 12, 18, 24, mean-pooled over real audio frames)

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperModel
WE_CACHE = 'cache_whisper_enc_v6.npy'
if os.path.exists(WE_CACHE): WE = np.load(WE_CACHE)
else:
    wfe = WhisperFeatureExtractor.from_pretrained('openai/whisper-medium')
    wenc = WhisperModel.from_pretrained('openai/whisper-medium', torch_dtype=torch.float16 if DEVICE == 0 else torch.float32).encoder.eval().to(dev)
    WLAY = (12, 18, 24)
    @torch.no_grad()
    def whisper_enc(p):
        y_ = load(p); chunks = [y_[i:i+30*SR] for i in range(0, len(y_), 30*SR)]
        chunks = [c for c in chunks if len(c) > SR] or [y_]
        out = []
        for c in chunks:
            x = wfe(c, sampling_rate=SR, return_tensors='pt').input_features.to(dev).to(wenc.dtype)
            hs = wenc(x, output_hidden_states=True).hidden_states
            nf = max(1, int(len(c) / SR * 50))                       # 50 encoder frames / second; skip padded part
            out.append(torch.cat([hs[l][0, :nf].float().mean(0) for l in WLAY]).cpu().numpy())
        return np.mean(out, 0)
    WE = np.stack([whisper_enc(p) for p in tqdm(pd.concat([train.path, test.path]), desc='Whisper encoder')])
    np.save(WE_CACHE, WE); del wenc; torch.cuda.empty_cache()
WE_tr, WE_te = WE[:len(train)], WE[len(train):]
print(WE.shape)

### 5c. Prompted Whisper transcripts
A disfluency/error-preserving prompt makes Whisper keep um/uh and ungrammatical speech that plain Whisper silently 'fixes'. Text features and sentence embeddings are computed for these second transcripts and merged with the originals (leaderboard 0.3814 -> 0.3800).

In [ ]:
import shutil, re
from transformers import WhisperProcessor, WhisperForConditionalGeneration
CACHE_P = 'cache_text_prompted_v7.csv'
PROMPT = "Umm, let me think like, hmm... Okay, so, uh, I go to the school yesterday and, um, we was playing with my friends, you know."
if os.path.exists(CACHE_P):
    txp = pd.read_csv(CACHE_P).fillna('')
else:
    proc = WhisperProcessor.from_pretrained('openai/whisper-medium')
    wm = WhisperForConditionalGeneration.from_pretrained('openai/whisper-medium',
            torch_dtype=torch.float16 if DEVICE == 0 else torch.float32).to(dev).eval()
    prompt_ids = proc.get_prompt_ids(PROMPT, return_tensors='pt').to(dev)
    def clean(t): return re.sub(r'\s+', ' ', t.replace(PROMPT, '')).strip()
    paths, outs, GB = pd.concat([train.path, test.path]).tolist(), [], 4
    for i in tqdm(range(0, len(paths), GB), desc='Whisper (prompted)'):
        chunks, owner = [], []
        for j, p in enumerate(paths[i:i+GB]):
            y_ = load(p); cs = [y_[k:k+30*SR] for k in range(0, len(y_), 30*SR)]
            cs = [c for c in cs if len(c) > SR] or [y_]
            chunks += cs; owner += [j] * len(cs)
        feats = proc.feature_extractor(chunks, sampling_rate=SR, return_tensors='pt').input_features.to(dev).to(wm.dtype)
        with torch.no_grad():
            ids = wm.generate(feats, prompt_ids=prompt_ids, language='en', task='transcribe', max_new_tokens=220)
        txt = [clean(t) for t in proc.batch_decode(ids, skip_special_tokens=True)]
        for j in range(len(paths[i:i+GB])):
            outs.append(' '.join(t for t, o in zip(txt, owner) if o == j).strip())
    txp = pd.DataFrame({'text': outs}); txp.to_csv(CACHE_P, index=False)
    del wm; torch.cuda.empty_cache()
train['text_p'] = txp.text.values[:len(train)]; test['text_p'] = txp.text.values[len(train):]
print(train[['text', 'text_p']].head(3).to_string())


In [ ]:
import language_tool_python
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
TXT_CACHE_P = 'cache_textfeats_prompted_v7.csv'
if os.path.exists(TXT_CACHE_P):
    TF2 = pd.read_csv(TXT_CACHE_P, index_col=0)
else:
    try: tool = language_tool_python.LanguageTool('en-US')
    except Exception as e: tool = None; print('LanguageTool unavailable:', e)
    cola = pipeline('text-classification', model='textattack/roberta-base-CoLA', device=DEVICE, top_k=None, truncation=True, max_length=128)
    tok = AutoTokenizer.from_pretrained('gpt2'); lm = AutoModelForCausalLM.from_pretrained('gpt2').eval().to(dev)
    FILLERS = {'uh', 'um', 'umm', 'uhh', 'er', 'erm', 'ah', 'hmm'}
    def sentences(t):
        s = [x.strip() for x in re.split(r'(?<=[.!?])\s+', t) if len(x.split()) >= 2]
        return s or [t if t else 'empty']
    @torch.no_grad()
    def ppl(t):
        ids = tok(t, return_tensors='pt', truncation=True, max_length=512).input_ids.to(dev)
        if ids.shape[1] < 2: return 0.0
        return float(torch.exp(lm(ids, labels=ids).loss).clamp(max=5000).log())
    def acc(res):
        d = {r['label']: r['score'] for r in res}; return d.get('LABEL_1', d.get('acceptable', 0.0))
    def text_feats(t):
        w = re.findall(r"[a-zA-Z']+", t.lower()); n_ = max(len(w), 1)
        ss = sentences(t); sl = [len(s.split()) for s in ss]; p = [acc(r) for r in cola(ss)]
        bg = list(zip(w, w[1:]))
        return dict(grammar_error_rate=(len(tool.check(t)) / n_) if tool else 0.0,
            n_words=n_, n_sent=len(ss), avg_sent_len=np.mean(sl), max_sent_len=max(sl), std_sent_len=np.std(sl),
            ttr=len(set(w)) / n_, filler_rate=sum(x in FILLERS for x in w) / n_,
            repeat_bigram=1 - len(set(bg)) / max(len(bg), 1), immediate_repeat=sum(a == b for a, b in bg) / n_,
            avg_word_len=np.mean([len(x) for x in w]) if w else 0,
            cola_mean=np.mean(p), cola_min=np.min(p), cola_frac_ok=np.mean(np.array(p) > 0.5),
            log_ppl=ppl(t), has_punct=float(bool(re.search(r'[.!?]', t))))
    TF2 = pd.DataFrame([text_feats(t) for t in tqdm(pd.concat([train.text_p, test.text_p]), desc='Text feats (prompted)')]).add_prefix('p_')
    TF2.to_csv(TXT_CACHE_P); del cola, lm; torch.cuda.empty_cache()
print(TF2.shape)


In [ ]:
from transformers import AutoModel
EMB_CACHE_P = 'cache_text_emb_prompted_v7.npy'
if os.path.exists(EMB_CACHE_P): TE2 = np.load(EMB_CACHE_P)
else:
    etok = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
    emod = AutoModel.from_pretrained('sentence-transformers/all-mpnet-base-v2').eval().to(dev)
    texts, out = pd.concat([train.text_p, test.text_p]).fillna('').tolist(), []
    with torch.no_grad():
        for i in range(0, len(texts), 16):
            b = etok(texts[i:i+16], padding=True, truncation=True, max_length=384, return_tensors='pt').to(dev)
            h = emod(**b).last_hidden_state; m = b['attention_mask'].unsqueeze(-1).float()
            out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    TE2 = np.vstack(out); np.save(EMB_CACHE_P, TE2); del emod; torch.cuda.empty_cache()

n0 = len(train)
TF_old = pd.read_csv('cache_textfeats_medium_final.csv', index_col=0)
TF = pd.concat([TF_old, TF2], axis=1)
X_txt_tr, X_txt_te = TF.iloc[:n0].values, TF.iloc[n0:].values
T_tr, T_te = np.hstack([TE[:n0], TE2[:n0]]), np.hstack([TE[n0:], TE2[n0:]])
print('text feats:', TF.shape, '| text emb:', T_tr.shape)


## 6. Modelling: base learners → OOF → stacking

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, HuberRegressor, LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

# ===== SWITCH =====  True: train without the 0-score clips (scored 0.4199 on the leaderboard)
#                     False: keep them + zero-gate classifier.  Run both, submit both, keep the better one.
DROP_ZEROS = True

y_all = train[LABEL].values.astype(float)
keep = (y_all > 0) if DROP_ZEROS else np.ones(len(y_all), bool)
y = y_all[keep]; n = len(y)
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))
X_hand_tr = np.hstack([X_txt_tr, X_pro_tr])[keep]; X_hand_te = np.hstack([X_txt_te, X_pro_te])
X_emb_tr, W_tr, T_tr, WE_tr = X_emb_tr[keep], W_tr[keep], T_tr[keep], WE_tr[keep]

Xs = {'w2v7': (X_emb_tr, X_emb_te), 'w2vML': (W_tr, W_te), 'txtemb': (T_tr, T_te), 'hand': (X_hand_tr, X_hand_te),
      'all7': (np.hstack([X_emb_tr, X_hand_tr]), np.hstack([X_emb_te, X_hand_te])),
      'combo': (np.hstack([W_tr, T_tr, X_hand_tr]), np.hstack([W_te, T_te, X_hand_te])),
      'wenc': (WE_tr, WE_te),
      'combo2': (np.hstack([WE_tr, W_tr, T_tr, X_hand_tr]), np.hstack([WE_te, W_te, T_te, X_hand_te]))}
A_BIG, A_MID = np.logspace(2, 5.5, 10), np.logspace(1, 4, 8)
ORIG = ['huber_w2v', 'rf_hand', 'ridge_hand', 'huber_all']          # the 4-model blend of the 0.4199 run
models = {
 'huber_w2v':    (lambda: make_pipeline(StandardScaler(), PCA(24, random_state=SEED), HuberRegressor(epsilon=1.35, max_iter=500)), 'w2v7'),
 'rf_hand':      (lambda: RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_split=5, random_state=SEED, n_jobs=-1), 'hand'),
 'ridge_hand':   (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=[1, 10, 50, 100])), 'hand'),
 'huber_all':    (lambda: make_pipeline(StandardScaler(), PCA(32, random_state=SEED), HuberRegressor(epsilon=1.35, max_iter=500)), 'all7'),
 'ridge_w2vML':  (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=A_BIG)), 'w2vML'),
 'svr_w2vML':    (lambda: make_pipeline(StandardScaler(), PCA(64, random_state=SEED), SVR(C=2, epsilon=0.3)), 'w2vML'),
 'ridge_txtemb': (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=A_MID)), 'txtemb'),
 'svr_txtemb':   (lambda: make_pipeline(StandardScaler(), PCA(48, random_state=SEED), SVR(C=2, epsilon=0.3)), 'txtemb'),
 'ridge_combo':  (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=A_BIG)), 'combo'),
 'ridge_wenc':   (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=A_BIG)), 'wenc'),
 'svr_wenc':     (lambda: make_pipeline(StandardScaler(), PCA(64, random_state=SEED), SVR(C=2, epsilon=0.3)), 'wenc'),
 'ridge_combo2': (lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=A_BIG)), 'combo2'),
 'gbr_hand':     (lambda: GradientBoostingRegressor(loss='huber', n_estimators=250, max_depth=3, learning_rate=0.03, subsample=0.8, random_state=SEED), 'hand'),
}
SEEDS = (42, 7, 2026)
oof = pd.DataFrame(0.0, index=range(n), columns=models)
for sd in SEEDS:
    for tr, va in KFold(5, shuffle=True, random_state=sd).split(np.arange(n)):
        for name, (mk, key) in models.items():
            Xtr = Xs[key][0]; oof.loc[va, name] += mk().fit(Xtr[tr], y[tr]).predict(Xtr[va]) / len(SEEDS)
for name in models:
    p = np.clip(oof[name], 0, 5); print(f'{name:13s} OOF RMSE={rmse(y,p):.4f}  Pearson={pearsonr(y,p)[0]:.4f}')

In [ ]:
# ---- Stacker: compare original-4 / v5 / full(+Whisper-encoder) blends by nested-CV RMSE, keep the best ----
def nested(cols):
    b = np.zeros(n)
    for tr, va in KFold(5, shuffle=True, random_state=1).split(oof):
        b[va] = LinearRegression(positive=True).fit(oof[cols].values[tr], y[tr]).predict(oof[cols].values[va])
    return np.clip(b, 0, 5)
NOENC = [m for m in models if 'wenc' not in m and m != 'ridge_combo2']     # = the v5 blend
cands = {'original4': ORIG, 'v5_blend': NOENC, 'full': list(models)}
scores = {k: rmse(y, nested(c)) for k, c in cands.items()}
print('nested-CV RMSE:', {k: round(v, 4) for k, v in scores.items()})
USE = min(scores, key=scores.get); cols = cands[USE]; print('using blend:', USE)
stack = LinearRegression(positive=True).fit(oof[cols].values, y)
print('weights:', dict(zip(cols, stack.coef_.round(3))), 'intercept', round(stack.intercept_, 3))
bl_oof = nested(cols)

# ---- zero gate (only when zero clips are kept) ----
GATE = None
if not DROP_ZEROS:
    is0 = (y == 0).astype(int)
    gate_clf = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000, class_weight='balanced'))
    p0 = cross_val_predict(gate_clf(), X_hand_tr, is0, cv=KFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
    best = rmse(y, bl_oof)
    for t in (0.5, 0.6, 0.7, 0.8, 0.9):
        r = rmse(y, np.clip(np.where(p0 > t, 0.0, bl_oof), 0, 5))
        if r < best - 1e-4: best, GATE = r, t
    print('zero-gate threshold:', GATE)
    if GATE is not None: bl_oof = np.where(p0 > GATE, 0.0, bl_oof)
bl_oof = np.clip(bl_oof, 0, 5)
CV_RMSE, CV_PEARSON = rmse(y, bl_oof), pearsonr(y, bl_oof)[0]

# ---- refit on ALL training data -> train/test predictions ----
tr_p, te_p = {}, {}
for name in cols:
    mk, key = models[name]; m = mk().fit(Xs[key][0], y); tr_p[name] = m.predict(Xs[key][0]); te_p[name] = m.predict(Xs[key][1])
train_pred = stack.predict(pd.DataFrame(tr_p)[cols].values); test_pred = stack.predict(pd.DataFrame(te_p)[cols].values)
if GATE is not None:
    gc = gate_clf().fit(X_hand_tr, is0)
    train_pred = np.where(gc.predict_proba(X_hand_tr)[:, 1] > GATE, 0.0, train_pred)
    test_pred  = np.where(gc.predict_proba(X_hand_te)[:, 1] > GATE, 0.0, test_pred)
train_pred, test_pred = np.clip(train_pred, 0, 5), np.clip(test_pred, 0, 5)
TRAIN_RMSE = rmse(y, train_pred)
print('=' * 55)
print('DROP_ZEROS =', DROP_ZEROS, '| blend =', USE)
print(f'TRAINING RMSE (required) : {TRAIN_RMSE:.4f}')
print(f'Training Pearson         : {pearsonr(y, train_pred)[0]:.4f}')
print(f'5-fold CV RMSE / Pearson : {CV_RMSE:.4f} / {CV_PEARSON:.4f}')
print('=' * 55)

## 7. Visualizations & interpretation

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].scatter(y, bl_oof, alpha=.5); ax[0].plot([0, 5], [0, 5], 'r--')
ax[0].set(xlabel='True', ylabel='Predicted (OOF)', title=f'OOF: RMSE={CV_RMSE:.3f}, r={CV_PEARSON:.3f}')
sns.histplot(y - bl_oof, bins=25, kde=True, ax=ax[1]); ax[1].set_title('Residuals (OOF)')
sns.kdeplot(y, ax=ax[2], label='train true'); sns.kdeplot(test_pred, ax=ax[2], label='test pred'); ax[2].legend(); ax[2].set_title('Score distributions')
plt.tight_layout(); plt.show()
pd.DataFrame({'true': y.round(), 'abs_err': np.abs(y - bl_oof)}).groupby('true').abs_err.mean().plot.bar(title='Mean abs error by true score'); plt.show()
pro_names = ['dur', 'speech_ratio', 'seg_rate', 'mean_gap', 'max_gap', 'rms']
names = list(TF.columns)[:X_hand_tr.shape[1] - len(pro_names)] + pro_names
g = GradientBoostingRegressor(n_estimators=250, max_depth=3, learning_rate=.03, subsample=.8, random_state=SEED).fit(X_hand_tr, y)
pd.Series(g.feature_importances_, index=names).sort_values().plot.barh(figsize=(6, 6), title='Feature importance (hand-crafted)'); plt.show()
pd.DataFrame(X_hand_tr, columns=names).assign(label=y).corr()['label'].drop('label').sort_values().plot.barh(figsize=(6, 6), title='Correlation with grammar score'); plt.show()
pd.Series(dict(zip(cols, stack.coef_))).plot.barh(title='Stacking weights'); plt.show()

## 8. Qualitative check: transcripts per score

In [ ]:
for s in [1, 2, 3, 4, 5]:
    r = train[train[LABEL].round() == s].head(1)
    if len(r): print(f'--- score {r[LABEL].values[0]} ---\n{r.text.values[0][:400]}\n')

## 9. Submission files

In [ ]:
from IPython.display import FileLink, display
out = test_csv[[FNAME]].copy(); out[LABEL] = np.round(test_pred, 4)
assert out[LABEL].notna().all() and len(out) == len(test_csv) and list(out.columns) == list(sub.columns)
out.to_csv('submission.csv', index=False)
out.to_csv('submission_dropzero.csv' if DROP_ZEROS else 'submission_keepzero.csv', index=False)
print(out.head(), out.shape)
print(f'FINAL -> Train RMSE: {TRAIN_RMSE:.4f} | CV RMSE: {CV_RMSE:.4f} | CV Pearson: {CV_PEARSON:.4f}')
display(FileLink('submission.csv'))

## 10. Conclusion
* Pipeline: Whisper-medium → (LanguageTool + CoLA + GPT-2 + fluency stats + sentence embedding) and wav2vec2 (layer 7 + multi-layer) + prosody → Ridge/SVR/Huber/RF/GBR → non-negative stack → clip [0, 5].
* Reported: **training RMSE** and **5-fold CV RMSE / Pearson** (two split seeds).
* **Future work:** fine-tune wav2vec2/Whisper encoder with a regression head, ordinal loss, seed ensembling, pseudo-labelling on test.